In [22]:
import os
import random
import pandas as pd
import numpy as np
from pathlib import Path
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from PatchDataset import load_dataset
from pipeline import set_seed


PATCH_SIZE = 224
BATCH_SIZE = 32
SEED = 42
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

PATCHES_ROOT = "patches_dataset/patches_v3_seed42_pad1.6_iouth0.09"
traincsv_path = os.path.join(PATCHES_ROOT, "metadata.csv")
testcsv_path = "patches_dataset/test_patches_v3/test_metadata.csv"

AUG_MULTIPLIER = 2.0  # x2 training size
TARGET_POS_RATIO = 0.60


cuda


In [23]:
traindf = pd.read_csv(traincsv_path)
traindf["patch_id"] = traindf["patch_id"].astype(int)
traindf["label"] = traindf["label"].astype(int)
traindf["type"] = traindf["type"].astype(str)

testdf = pd.read_csv(testcsv_path)
testdf["patch_id"] = testdf["patch_id"].astype(int)
testdf["label"] = testdf["label"].astype(int)
testdf["type"] = testdf["type"].astype(str)

train_dataset,val_dataset,train_loader,val_loader,test_dataset,test_loader = load_dataset(traindf,testdf,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break


train/val: 15858 3965
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0) tensor(1)


In [24]:
print(f"Train: {len(traindf)} | Test: {len(testdf)}")
print("Train balance:", traindf["label"].value_counts(normalize=True))
print("Test balance:", testdf["label"].value_counts(normalize=True))

Train: 19823 | Test: 3965
Train balance: label
1    0.684911
0    0.315089
Name: proportion, dtype: float64
Test balance: label
1    0.684994
0    0.315006
Name: proportion, dtype: float64


In [25]:
AUG_ROOT = "patches_dataset/patches_v4_train_aug"
os.makedirs(os.path.join(AUG_ROOT, "positive"), exist_ok=True)
os.makedirs(os.path.join(AUG_ROOT, "negative"), exist_ok=True)

print("Augmented directories created")

Augmented directories created


In [26]:
train_aug_pos = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.08, rotate_limit=20, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
])

train_aug_neg = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.06, rotate_limit=15, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
])

/home/alant/myglobal/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [28]:
train_aug_pos = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.OneOf([
        A.HorizontalFlip(p=1.0),
        A.VerticalFlip(p=1.0),
        A.RandomRotate90(p=1.0),
        A.Transpose(p=1.0),
    ], p=0.9),
    A.OneOf([
        A.Affine(scale=(0.80, 1.20), rotate=(-30, 30), shear=(-8, 8), p=1.0),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=30, p=1.0),
    ], p=0.8),
    A.OneOf([
        A.RandomBrightnessContrast(0.25, 0.25, p=1.0),
        A.HueSaturationValue(8, 20, 15, p=1.0),
        A.CLAHE(clip_limit=2.0, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
    ], p=0.35),
    A.OneOf([
        A.MotionBlur(blur_limit=5, p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
    ], p=0.2),
])

train_aug_neg = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.OneOf([
        A.HorizontalFlip(p=1.0),
        A.VerticalFlip(p=1.0),
        A.RandomRotate90(p=1.0),
    ], p=0.8),
    A.OneOf([
        A.Affine(scale=(0.85, 1.15), rotate=(-20, 20), shear=(-5, 5), p=1.0),
        A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.10, rotate_limit=20, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.RandomBrightnessContrast(0.20, 0.20, p=1.0),
        A.HueSaturationValue(6, 15, 12, p=1.0),
    ], p=0.6),
    A.OneOf([
        A.GaussNoise(var_limit=(15.0, 70.0), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.1, 0.4), p=1.0),
    ], p=0.35),
])

/tmp/ipykernel_75254/1646658006.py:19: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
/tmp/ipykernel_75254/1646658006.py:44: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(15.0, 70.0), p=1.0),


In [29]:
def generate_augmented_samples(df_train, aug_root, multiplier=2.0, pos_ratio=0.60):
    train_pos = df_train[df_train["label"] == 1].copy()
    train_neg = df_train[df_train["label"] == 0].copy()
    
    P, N = len(train_pos), len(train_neg)
    T = int(multiplier * (P + N))
    target_pos, target_neg = int(pos_ratio * T), T - int(pos_ratio * T)
    
    aug_pos_needed = max(0, target_pos - P)
    aug_neg_needed = max(0, target_neg - N)
    
    print(f"Original train: {P+N} (P:{P}, N:{N})")
    print(f"Target: {T} (P:{target_pos}, N:{target_neg})")
    print(f"Need to augment: pos={aug_pos_needed}, neg={aug_neg_needed}")
    
    aug_metadata = []
    
    # Augment positives
    pos_dir_orig = os.path.join(PATCHES_ROOT, "positive")
    pos_dir_aug = os.path.join(aug_root, "positive")
    
    for i in range(aug_pos_needed):
        orig_idx = i % len(train_pos) # so that it doesnt go out of bounds
        orig_patch_id = train_pos.iloc[orig_idx]["patch_id"]
        
        img_path = os.path.join(pos_dir_orig, f"{orig_patch_id}.png")
        img = cv2.imread(img_path)
        if img is None: 
            print(f"Warning: Could not read image {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        aug_img = train_aug_pos(image=img)["image"]
        new_pid = f"patch{orig_patch_id}_aug_{i}"
        cv2.imwrite(os.path.join(pos_dir_aug, f"{new_pid}.png"), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
        
        aug_metadata.append({
            "patch_id": new_pid,
            "label": 1,
            "type": "positive_aug"
        })
    
    # Augment negatives
    neg_dir_orig = os.path.join(PATCHES_ROOT, "negative")
    neg_dir_aug = os.path.join(aug_root, "negative")
    
    for i in range(aug_neg_needed):
        orig_idx = i % len(train_neg)
        orig_patch_id = train_neg.iloc[orig_idx]["patch_id"]
        
        img_path = os.path.join(neg_dir_orig, f"{orig_patch_id}.png")
        img = cv2.imread(img_path)
        if img is None: 
            print(f"Warning: Could not read image {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        aug_img = train_aug_neg(image=img)["image"]
        new_pid = f"patch{orig_patch_id}_aug_{i}"
        cv2.imwrite(os.path.join(neg_dir_aug, f"{new_pid}.png"), 
                   cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
        
        aug_metadata.append({
            "patch_id": new_pid,
            "label": 0,
            "type": "negative_aug"
        })
    
    aug_df = pd.DataFrame(aug_metadata)
    return aug_df


In [30]:
# Generate augmented training data
train_aug_df = generate_augmented_samples(traindf, AUG_ROOT)
print(f"Generated {len(train_aug_df)} augmented samples")

Original train: 19823 (P:13577, N:6246)
Target: 39646 (P:23787, N:15859)
Need to augment: pos=10210, neg=9613
Generated 19823 augmented samples


In [31]:
# dropping the columns cx and cy and filename
train_df_copy = traindf.copy().drop(columns=["cx", "cy", "filename"])
train_df_combined = pd.concat([
    train_df_copy,
    train_aug_df
], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Combined training dataset:")
print(train_df_combined["label"].value_counts())
print(train_df_combined["label"].value_counts(normalize=True))
df_combined_path = os.path.join(AUG_ROOT, "metadata.csv")
train_df_combined.to_csv(df_combined_path, index=False)

Combined training dataset:
label
1    23787
0    15859
Name: count, dtype: int64
label
1    0.599985
0    0.400015
Name: proportion, dtype: float64


In [32]:
test_path = "patches_dataset/test_patches_v3/test_metadata.csv"
testdf.to_csv(test_path, index=False)
print("Test dataset (original, unchanged):")
print(testdf["label"].value_counts())

Test dataset (original, unchanged):
label
1    2716
0    1249
Name: count, dtype: int64


In [18]:
def validate_dataset(root_dir):    
    root = Path(root_dir)
    for subdir in ['positive', 'negative']:
        dir_path = root / subdir    
        for img_path in dir_path.glob("*.png"):
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"FAILED to read {img_path}")
                return False
    print("all good")

validate_dataset("patches_dataset/patches_v3_seed42_pad1.6_iouth0.09")

validate_dataset("patches_dataset/patches_v3_train_aug")

all good
all good
